In [18]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
import functools
from concurrent.futures import ThreadPoolExecutor
import time
import pyodbc
import pandas as pd
import os, json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import requests
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from tqdm import tqdm
import nest_asyncio
import uvicorn

In [19]:
import pyodbc
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get credentials from environment variables (SECURE)
SERVER = os.getenv("DB_SERVER", "tcp:swp391-sp26-server.database.windows.net")
DATABASE = os.getenv("DB_NAME", "SWP391_SP26_Group1")
USERNAME = os.getenv("DB_USER", "SWP391_SP26_Group1")
PASSWORD = os.getenv("DB_PASSWORD", "Team11111111")

if not PASSWORD:
    print("⚠️ WARNING: DB_PASSWORD not set in .env file!")

def get_conn():
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={SERVER},1433;"
        f"DATABASE={DATABASE};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
    )
    return pyodbc.connect(conn_str)

# Test connection
try:
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT TOP 5 * FROM dbo.SystemErrorLog;")
    for row in cursor.fetchall():
        print(row)
    conn.close()
except Exception as e:
    print(f"❌ Connection error: {e}")


('00074562-9970-49ea-ac25-25dda6782f21', 500, 'ArgumentException', "The 'ClientId' option must be provided. (Parameter 'ClientId') | Inner Cause: The 'ClientId' option must be provided. (Parameter 'ClientId')", '   at Microsoft.AspNetCore.Authentication.OAuth.OAuthOptions.Validate()\r\n   at Microsoft.AspNetCore.Authentication.RemoteAuthenticationOptions.Validate(String scheme)\r\n   at Microsoft.AspNetCore.Authentication.AuthenticationBuilder.<>c__DisplayClass4_0`2.<AddSchemeHelper>b__1(TOptions o)\r\n   at Microsoft.Extensions.Options.ValidateOptions`1.Validate(String name, TOptions options)\r\n   at Microsoft.Extensions.Options.OptionsFactory`1.Create(String name)\r\n   at Microsoft.Extensions.Options.OptionsMonitor`1.<>c.<Get>b__10_0(String name, IOptionsFactory`1 factory)\r\n   at Microsoft.Extensions.Options.OptionsCache`1.<>c__DisplayClass3_1`1.<GetOrAdd>b__2()\r\n   at System.Lazy`1.ViaFactory(LazyThreadSafetyMode mode)\r\n--- End of stack trace from previous location ---\r\n  

In [20]:
from concurrent.futures import ThreadPoolExecutor

In [21]:
#Knowledge Base

In [22]:

def load_events_bundle():
    """Load events, agendas, docs, and locations from database."""
    
    # 1) Event
    events = pd.read_sql("""
        SELECT
            e.Id AS EventId,
            e.Title,
            e.Description,
            e.StartTime,
            e.EndTime,
            e.Status,
            e.DepartmentId,
            e.SemesterId,
            e.UpdatedAt
        FROM Event e
        WHERE (e.DeletedAt IS NULL OR e.DeletedAt = '') 
          AND e.Status IN ('Published')  
    """, get_conn())

    # 2) EventAgenda
    agenda = pd.read_sql("""
        SELECT
            a.Id AS AgendaId,
            a.EventId,
            a.SessionName,
            a.Description AS SessionDescription,
            a.SpeakerName,
            a.StartTime AS SessionStart,
            a.EndTime AS SessionEnd
        FROM EventAgenda a
        WHERE (a.DeletedAt IS NULL)
    """, get_conn())

    # 3) EventDocuments
    docs = pd.read_sql("""
        SELECT
            d.Id AS DocId,
            d.EventId,
            d.Name AS DocTitle,
            d.Url
        FROM EventDocument d
        WHERE (d.DeletedAt IS NULL)
    """, get_conn())
    
    # 4) Locations
    locations = pd.read_sql("""
        SELECT
           l.Id        AS LocationId,
           l.Name      AS LocationName,
           l.Address,
           l.Capacity,
           l.Status,
           l.Type,
           l.Description,
           l.CreatedAt,
           l.UpdatedAt
        FROM Locations l
        WHERE (l.DeletedAt IS NULL)
    """, get_conn())

    return events, agenda, docs, locations

def load_logs(days=14):
    """Load system error logs from database."""
    logs = pd.read_sql("""
        SELECT
            sel.Id AS LogId,
            sel.StatusCode,
            sel.ExceptionType,
            sel.ExceptionMessage,
            sel.StackTrace,
            sel.Source,
            sel.UserId,
            sel.CreatedAt
        FROM SystemErrorLog sel
        WHERE sel.CreatedAt IS NOT NULL
    """, get_conn())
    return logs


In [23]:
#get role DB

In [24]:
def get_user_role(db, user_id: str) -> str:
    row = db.execute("""
        SELECT r.RoleName
        FROM [User] u
        JOIN Role r ON u.RoleId = r.Id
        WHERE u.Id = :user_id
    """, {"user_id": user_id}).fetchone()

    return row[0].lower() if row else "user"


In [25]:
def can_access(meta: dict, role: str):
    role = (role or "user").lower()

    mtype = (meta.get("type") or "").lower()
    access = (meta.get("access") or "").lower()

    # log hệ thống: CHỈ admin
    if mtype == "system_log":
        return role == "admin"

    # admin xem được hết
    if role == "admin":
        return True

    # user chỉ xem access=user
    return access == "user"


In [26]:
import sys
print(sys.executable)


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\python.exe


In [27]:
def build_event_text(event_row, agenda_rows, doc_rows, locations_rows=None):
    lines = []
    lines.append("EVENT OVERVIEW")
    lines.append(f"Title: {event_row.get('Title','')}")
    lines.append(f"Description: {event_row.get('Description','')}")
    lines.append(f"Time: {event_row.get('StartTime','')} - {event_row.get('EndTime','')}")
    lines.append(f"Status: {event_row.get('Status','')}")
    lines.append("")

    if len(agenda_rows) > 0:
        lines.append("AGENDA")
        for _, a in agenda_rows.iterrows():
            lines.append(f"- Session: {a.get('SessionName','')}")
            lines.append(f"  Speaker: {a.get('SpeakerName','')}")
            lines.append(f"  Time: {a.get('SessionStart','')} - {a.get('SessionEnd','')}")
            lines.append(f"  Detail: {a.get('SessionDescription','')}")
        lines.append("")

    if len(doc_rows) > 0:
        lines.append("DOCUMENTS")
        for _, d in doc_rows.iterrows():
            lines.append(f"- {d.get('DocTitle','')}")
            if d.get("Url"):
                lines.append(f"  URL: {d.get('Url')}")
        lines.append("")

    # locations_rows có thể None nếu bạn chưa join theo EventId
    if locations_rows is not None and len(locations_rows) > 0:
        lines.append("LOCATION DETAILS")
        for _, l in locations_rows.iterrows():
            lines.append(f"- {l.get('LocationName','')}")
            if l.get("Address"):
                lines.append(f"  Address: {l.get('Address')}")
            if l.get("Capacity"):
                lines.append(f"  Capacity: {l.get('Capacity')}")
            if l.get("Type"):
                lines.append(f"  Type: {l.get('Type')}")
        lines.append("")

    return "\n".join(lines).strip()


In [28]:
# Build KB logs (admin) - optimized: shorten stacktrace + enable roll-up summaries
import re

def _short_stack(s: str, max_lines: int = 15, max_chars: int = 1200) -> str:
    s = (s or "").strip()
    if not s:
        return ""
    lines = s.splitlines()[:max_lines]
    out = "\n".join(lines)
    return out[:max_chars]

def build_systemlog_text(r):
    return "\n".join([
        "SYSTEM ERROR LOG",
        f"StatusCode: {r.get('StatusCode','')}",
        f"ExceptionType: {r.get('ExceptionType','')}",
        f"ExceptionMessage: {r.get('ExceptionMessage','')}",
        f"Source: {r.get('Source','')}",
        f"UserId: {r.get('UserId','')}",
        f"CreatedAt: {r.get('CreatedAt','')}",
        f"StackTrace: {_short_stack(r.get('StackTrace',''))}",
    ])

# ---- Log roll-up (IMPORTANT for admin: prevents log-chunk flooding) ----
def _normalize_msg(s: str) -> str:
    s = (s or "").lower()
    # replace guids/hashes/numbers that cause duplicates to look different
    s = re.sub(r"\b[0-9a-f]{8,}\b", "<id>", s)
    s = re.sub(r"\b\d+\b", "<n>", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:400]

def rollup_logs(logs_df: pd.DataFrame) -> pd.DataFrame:
    """Group similar errors into summary rows (Count/FirstAt/LastAt)."""
    if logs_df is None or len(logs_df) == 0:
        return logs_df

    df = logs_df.copy()
    df["MsgNorm"] = df["ExceptionMessage"].apply(_normalize_msg)
    df["Sig"] = (
        df["ExceptionType"].fillna("").astype(str) + "|" +
        df["Source"].fillna("").astype(str) + "|" +
        df["StatusCode"].fillna("").astype(str) + "|" +
        df["MsgNorm"].fillna("").astype(str)
    )

    g = df.groupby("Sig", as_index=False).agg(
        Count=("Sig", "size"),
        FirstAt=("CreatedAt", "min"),
        LastAt=("CreatedAt", "max"),
        ExceptionType=("ExceptionType", "first"),
        ExceptionMessage=("ExceptionMessage", "first"),
        Source=("Source", "first"),
        StatusCode=("StatusCode", "first"),
    )
    return g.sort_values(["Count", "LastAt"], ascending=[False, False])


In [29]:
def build_all(days_logs: int = 14):
    """Build both KB and LOG indexes from database."""
    print("Building indexes...")

    events, agenda, docs, locations = load_events_bundle()
    logs = load_logs(days=days_logs)
    logs_sum = rollup_logs(logs)

    # ---- KB ----
    kb_texts, kb_meta = [], []
    for _, e in events.iterrows():
        eid = e["EventId"]
        text = build_event_text(e, agenda[agenda["EventId"]==eid], docs[docs["EventId"]==eid], None)
        chunks = chunk_text(text)
        for i, ch in enumerate(chunks):
            kb_texts.append(ch)
            kb_meta.append({
                "type":"event",
                "access":"user",
                "event_id": str(eid),
                "chunk_id": i
            })

    kb_index = build_index(kb_texts)
    faiss.write_index(kb_index, "kb/faiss_kb.index")

    # ---- LOG ----
    log_texts, log_meta = [], []
    for _, r in logs_sum.iterrows():
        text = f"""
SYSTEM ERROR SUMMARY
StatusCode: {r.get('StatusCode','')}
ExceptionType: {r.get('ExceptionType','')}
Source: {r.get('Source','')}
Message: {r.get('ExceptionMessage','')}
Count: {r.get('Count','')}
FirstAt: {r.get('FirstAt','')}
LastAt: {r.get('LastAt','')}
"""
        log_texts.append(text)
        log_meta.append({"type": "system_log_summary", "access": "admin"})

    log_index = build_index(log_texts) if log_texts else None
    if log_index:
        faiss.write_index(log_index, "kb/faiss_log.index")

    print("Done building.")
    return kb_index, log_index


In [30]:
#chunking context
from typing import List

def chunk_text(text: str, max_chars: int = 3000, overlap: int = 250) -> List[str]:
    """Split text into overlapping chunks."""
    chunks = []
    i = 0
    while i < len(text):
        j = min(len(text), i + max_chars)
        chunks.append(text[i:j])
        i = j - overlap
        if i < 0:
            i = 0
        if j == len(text):
            break
    return chunks

# ===== LOAD EMBEDDING MODEL =====
print("Loading SentenceTransformer model...")
try:
    model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast & lightweight
    print(f"✅ Model loaded: {model.get_sentence_embedding_dimension()} dimensions")
except Exception as e:
    print(f"❌ Model loading error: {e}")
    model = None

# ===== BUILD INDEX FUNCTION =====
def build_index(texts: List[str]) -> faiss.IndexFlatIP:
    """Build FAISS index from text chunks."""
    if not texts:
        raise ValueError("No texts provided to build index")
    
    if model is None:
        raise RuntimeError("Model not loaded")
    
    print(f"Encoding {len(texts)} texts...")
    passages = [f"passage: {t}" for t in texts]
    embs = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
    embs = np.asarray(embs, dtype="float32")
    
    if embs.ndim == 1:
        embs = embs.reshape(1, -1)
    
    dim = embs.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(embs)
    
    print(f"✅ Index built: {len(texts)} chunks, {dim} dimensions")
    return idx

# ===== GROQ API ENDPOINT =====
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"


Loading SentenceTransformer model...
✅ Model loaded: 384 dimensions


In [31]:
import os

PROJECT_DIR = r"D:\PROJECT_SWP_SP26\Python"   # đúng folder project của bạn
os.chdir(PROJECT_DIR)

KB_DIR = os.path.join(PROJECT_DIR, "kb")
os.makedirs(KB_DIR, exist_ok=True)

print("CWD =", os.getcwd())
print("KB exists?", os.path.exists(KB_DIR))
print("KB files:", os.listdir(KB_DIR))


CWD = D:\PROJECT_SWP_SP26\Python
KB exists? True
KB files: ['chunks.jsonl', 'chunks_kb.jsonl', 'chunks_log.jsonl', 'faiss.index', 'faiss_kb.index', 'faiss_log.index']


In [32]:
import os, faiss

kb_path  = os.path.join(KB_DIR, "faiss_kb.index")
log_path = os.path.join(KB_DIR, "faiss_log.index")

if not os.path.exists(kb_path) or not os.path.exists(log_path):
    print("⚠️ Index not found → building...")
    kb_index, log_index = build_all(days_logs=14)
else:
    print("✅ Loading existing index...")
    kb_index  = faiss.read_index(kb_path)
    log_index = faiss.read_index(log_path)

print("KB ntotal =", kb_index.ntotal)
print("LOG ntotal =", log_index.ntotal)


✅ Loading existing index...
KB ntotal = 1
LOG ntotal = 21


In [33]:
def _build_and_save_index(texts: List[str], metas: list, index_path: str, jsonl_path: str) -> faiss.IndexFlatIP:
    """Build and save FAISS index + metadata to JSONL."""
    if len(texts) == 0:
        raise ValueError(f"No texts to build index: {index_path}")

    idx = build_index(texts)

    os.makedirs("kb", exist_ok=True)
    faiss.write_index(idx, index_path)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for t, m in zip(texts, metas):
            f.write(json.dumps({"text": t, "meta": m}, ensure_ascii=False) + "\n")

    print(f"✅ Saved {index_path}: {len(texts)} chunks")
    return idx


def main(days_logs_rollup: int = 14):
    """Main pipeline: load data → chunk → encode → index."""
    print("📊 Loading data from database...")
    events, agenda, docs, locations = load_events_bundle()
    logs = load_logs(days=days_logs_rollup)

    # Filter logs by date
    if "CreatedAt" in logs.columns:
        try:
            cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=days_logs_rollup)
            logs = logs[logs["CreatedAt"] >= cutoff]
        except Exception as e:
            print(f"⚠️ Date filtering error: {e}")

    print(f"Events: {len(events)} | Logs: {len(logs)}")

    # 1) Knowledge base chunks (events/docs)
    print("📝 Building KB chunks...")
    kb_chunks_data, kb_meta_data = [], []
    for _, e in events.iterrows():
        eid = e["EventId"]
        a_rows = agenda[agenda["EventId"] == eid]
        d_rows = docs[docs["EventId"] == eid]
        doc_text = build_event_text(e, a_rows, d_rows, locations_rows=locations)
        chunks = chunk_text(doc_text)

        for idx, ch in enumerate(chunks):
            ch = (ch or "").strip()
            if not ch:
                continue

            kb_chunks_data.append(ch)
            kb_meta_data.append({
                "type": "event",
                "access": "user",
                "event_id": str(eid),
                "chunk_id": idx,
                "department_id": str(e.get("DepartmentId", "")),
                "semester_id": str(e.get("SemesterId", "")),
                "status": str(e.get("Status", "")),
                "updated_at": str(e.get("UpdatedAt", "")),
                "source": "event_bundle"
            })

    # 2) Admin log chunks (roll-up summary)
    print("📋 Building LOG summary chunks...")
    log_sum = rollup_logs(logs)

    log_chunks_data, log_meta_data = [], []
    if log_sum is not None and len(log_sum) > 0:
        for _, r in log_sum.iterrows():
            text = "\n".join([
                "SYSTEM ERROR SUMMARY",
                f"StatusCode: {r.get('StatusCode','')}",
                f"ExceptionType: {r.get('ExceptionType','')}",
                f"Source: {r.get('Source','')}",
                f"Message: {r.get('ExceptionMessage','')}",
                f"Count: {r.get('Count','')}",
                f"FirstAt: {r.get('FirstAt','')}",
                f"LastAt: {r.get('LastAt','')}",
            ])
            log_chunks_data.append(text)
            log_meta_data.append({
                "type": "system_log_summary",
                "access": "admin",
                "status_code": str(r.get("StatusCode", "")),
                "exception_type": str(r.get("ExceptionType", "")),
                "source": str(r.get("Source", "")),
                "count": int(r.get("Count", 0)) if str(r.get("Count","")).isdigit() else r.get("Count", 0),
                "first_at": str(r.get("FirstAt", "")),
                "last_at": str(r.get("LastAt", "")),
                "source_kind": "log_rollup"
            })

    print(f"KB chunks: {len(kb_chunks_data)} | LOG chunks: {len(log_chunks_data)}")

    # Build and save indexes
    kb_index = _build_and_save_index(
        kb_chunks_data, kb_meta_data,
        index_path="kb/faiss_kb.index",
        jsonl_path="kb/chunks_kb.jsonl"
    )

    log_index = None
    if log_chunks_data:
        log_index = _build_and_save_index(
            log_chunks_data, log_meta_data,
            index_path="kb/faiss_log.index",
            jsonl_path="kb/chunks_log.jsonl"
        )

    return kb_index, log_index


print("Starting main build process...")
kb_index, log_index = main(days_logs_rollup=14)
print("✅ Build complete!")


Starting main build process...
📊 Loading data from database...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30824\404720421.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30824\404720421.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  agenda = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30824\404720421.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30824\404720421.py:47: UserWarning: pandas only supports SQLAlchemy connec

⚠️ Date filtering error: Invalid comparison between dtype=datetime64[ns] and Timestamp
Events: 1 | Logs: 569
📝 Building KB chunks...
📋 Building LOG summary chunks...
KB chunks: 1 | LOG chunks: 21
Encoding 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.97s/it]


✅ Index built: 1 chunks, 384 dimensions
✅ Saved kb/faiss_kb.index: 1 chunks
Encoding 21 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]

✅ Index built: 21 chunks, 384 dimensions
✅ Saved kb/faiss_log.index: 21 chunks
✅ Build complete!


In [34]:
# (moved) log debugging is now in chunks_kb.jsonl / chunks_log.jsonl


In [35]:
# (moved) print counts via the load cells below


In [36]:
#acess role

In [37]:
import json

kb_chunks = []
with open("kb/chunks_kb.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        kb_chunks.append(json.loads(line))

log_chunks = []
with open("kb/chunks_log.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        log_chunks.append(json.loads(line))

print("len(kb_chunks)  =", len(kb_chunks))
print("len(log_chunks) =", len(log_chunks))


len(kb_chunks)  = 1
len(log_chunks) = 21


In [38]:
LOG_HINTS = ["log", "error", "exception", "stack", "trace", "500", "statuscode", "crash", "lỗi", "bug"]

def looks_like_log_question(q: str) -> bool:
    q = (q or "").lower()
    return any(k in q for k in LOG_HINTS)

def _retrieve_from(index_obj, chunk_list, question: str, top_k: int = 5):
    q = model.encode([f"query: {question}"], normalize_embeddings=True)
    q = np.array(q, dtype="float32")

    # search rộng một chút để đủ lựa chọn
    k = min(index_obj.ntotal, max(top_k * 30, 30))
    scores, ids = index_obj.search(q, k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0 or idx >= len(chunk_list):
            continue
        item = chunk_list[idx]
        results.append({
            "score": float(score),
            "text": item["text"],
            "meta": item["meta"]
        })
        if len(results) >= top_k:
            break
    return results

def _is_too_generic(q: str) -> bool:
    q = (q or "").strip().lower()
    return q in ["có sự kiện nào không", "có sự kiện không", "sự kiện", "event", "có gì không"]

def sql_upcoming_published_events(limit: int = 5):
    # fallback for very generic user queries (bypass embeddings)
    try:
        df = pd.read_sql(f"""
            SELECT TOP ({limit})
                e.Title, e.StartTime, e.EndTime, e.Description
            FROM Event e
            WHERE (e.DeletedAt IS NULL OR e.DeletedAt = '') 
              AND e.Status IN ('Published')
            ORDER BY e.StartTime
        """, get_conn())
        return df
    except Exception:
        return pd.DataFrame()

def retrieve(question: str, top_k: int = 5, role: str = "user"):
    role = (role or "user").lower().strip()

    # USER: always search KB (events/docs)
    if role == "user":
        return _retrieve_from(kb_index, kb_chunks, question, top_k)

    # ADMIN: route to logs when question looks like troubleshooting/logs
    if looks_like_log_question(question):
        return _retrieve_from(log_index, log_chunks, question, top_k)

    # otherwise admin searching knowledge base
    return _retrieve_from(kb_index, kb_chunks, question, top_k)

def retrieve_with_fallback(question: str, top_k: int = 5, role: str = "user"):
    role = (role or "user").lower().strip()
    hits = retrieve(question, top_k=top_k, role=role)

    # If user asked generic "any events?" and retrieval is weak, return structured list
    if role == "user" and (_is_too_generic(question) or len(hits) == 0):
        df = sql_upcoming_published_events(limit=5)
        if len(df) > 0:
            # convert to pseudo hits so your LLM can cite them in context
            pseudo = []
            for i, r in df.iterrows():
                txt = f"EVENT\nTitle: {r.get('Title','')}\nStartTime: {r.get('StartTime','')}\nEndTime: {r.get('EndTime','')}\nDescription: {r.get('Description','')}"
                pseudo.append({
                    "score": 1.0,
                    "text": txt,
                    "meta": {"type":"event_sql", "access":"user", "source":"sql_fallback", "row": int(i)}
                })
            return pseudo

    return hits


In [39]:
import os
from dotenv import load_dotenv

# Reload .env file to get fresh API key
load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

if GROQ_API_KEY:
    print(f"✅ GROQ_API_KEY Loaded: {GROQ_API_KEY[:10]}...{GROQ_API_KEY[-10:]}")
    print(f"✅ Length: {len(GROQ_API_KEY)} characters")
else:
    print("❌ GROQ_API_KEY NOT SET!")


✅ GROQ_API_KEY Loaded: gsk_QEGTfW...XZei9nqI5n
✅ Length: 56 characters


In [40]:
# ===== VERIFY API KEY =====
import requests

def test_groq_api():
    """Quick test to verify Groq API key works."""
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        print("❌ No API key found")
        return False
    
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
    }
    
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": "Say 'OK' briefly"}],
        "temperature": 0.2,
        "max_tokens": 10,
    }
    
    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=10
        )
        
        if response.status_code == 200:
            print("✅ Groq API is working!")
            return True
        else:
            print(f"❌ API error {response.status_code}: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Connection error: {e}")
        return False

print("Testing Groq API...")
test_groq_api()


Testing Groq API...
✅ Groq API is working!


True

In [41]:
SYSTEM_PROMPT_USER = (
    "Bạn là trợ lý RAG cho hệ thống AEMS solution.\n"
    "CHỈ được trả lời dựa trên CONTEXT.\n"
    "KHÔNG dùng kiến thức bên ngoài.\n"
    "Nếu CONTEXT không có thông tin để trả lời, trả lời đúng câu:\n"
    "'Không tìm thấy thông tin trong dữ liệu.'\n"
    "Khi trả lời, TRÍCH DẪN ít nhất 1-2 mục [i] từ CONTEXT."
)

SYSTEM_PROMPT_ADMIN = (
    "Bạn là trợ lý RAG cho ADMIN.\n"
    "CHỈ được trả lời dựa trên CONTEXT.\n"
    "Ưu tiên phân tích LOG/ERROR/EXCEPTION nếu có.\n"
    "KHÔNG suy đoán ngoài dữ liệu.\n"
    "Nếu CONTEXT không có thông tin để trả lời, trả lời đúng câu:\n"
    "'Không tìm thấy thông tin trong dữ liệu.'\n"
    "Khi trả lời, TRÍCH DẪN ít nhất 1-2 mục [i] từ CONTEXT."
)

In [42]:
def _format_context(hits: list, max_chars_each: int = 900) -> str:
    """
    Format context có kèm meta để model bám theo dữ liệu.
    hits phần tử dạng: {"text":..., "meta":..., "score":...} (giống của bạn)
    """
    blocks = []
    for i, h in enumerate(hits, start=1):
        meta = h.get("meta", {}) or {}
        t = (h.get("text", "") or "").strip()
        if not t:
            continue

        # cắt bớt text để tránh quá dài
        if len(t) > max_chars_each:
            t = t[:max_chars_each] + "..."

        # lấy vài field quan trọng để trích dẫn
        ref_id = meta.get("event_id") or meta.get("log_id") or meta.get("location_id") or ""
        created_at = meta.get("created_at") or meta.get("updated_at") or ""
        m_type = meta.get("type", "")
        access = meta.get("access", "")

        blocks.append(
            f"[{i}] type={m_type} access={access} id={ref_id} time={created_at}\n{t}"
        )
    return "\n\n".join(blocks).strip()

In [43]:
def call_llm_groq(question: str, hits: list, role: str = "user", model_name="llama-3.1-8b-instant"):
    role = (role or "user").lower().strip()
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    # ✅ Lọc hits theo quyền truy cập bằng can_access(meta, role)
    filtered = []
    for h in (hits or []):
        meta = h.get("meta", {}) or {}
        if can_access(meta, role):
            filtered.append(h)

    # ✅ Nếu sau lọc không còn gì -> trả câu chuẩn (không gọi LLM cho đỡ tốn)
    if len(filtered) == 0:
        yield "Không tìm thấy thông tin trong dữ liệu."
        return

    context = _format_context(filtered)

    system_prompt = SYSTEM_PROMPT_ADMIN if role == "admin" else SYSTEM_PROMPT_USER

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}\n\n"
                f"QUESTION:\n{question}\n\n"
                "YÊU CẦU:\n"
                "- Trả lời ngắn gọn, đúng trọng tâm.\n"
                "- Chỉ dùng thông tin trong CONTEXT.\n"
                "- Luôn trích dẫn ít nhất 1-2 mục dạng [1], [2]...\n"
            )
        }
    ]

    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }

    payload = {
        "model": model_name,
        "messages": messages,
        "temperature": 0.2,
        "stream": True,
    }

    with requests.post(GROQ_URL, headers=headers, json=payload, stream=True, timeout=60) as r:
        r.encoding = "utf-8"
        if r.status_code != 200:
            raise RuntimeError(f"Groq error {r.status_code}: {r.text}")

        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            if line.startswith("data:"):
                data = line[len("data:"):].strip()
                if data == "[DONE]":
                    break
                try:
                    obj = json.loads(data)
                    delta = obj["choices"][0]["delta"].get("content")
                    if delta:
                        yield delta
                except Exception:
                    continue

In [44]:
app=FastAPI()

In [45]:
from pydantic import BaseModel


In [46]:
class AskReq(BaseModel):
    question: str
    top_k: int = 5
    role: str = ""


In [47]:
@app.get("/health")
def health():
    return {"status": "ok"}

In [48]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import nest_asyncio, uvicorn, threading

app = FastAPI(title="RAG API")

@app.post("/ask")  # JSON for .NET
def ask(req: AskReq):
    hits = retrieve_with_fallback(req.question, req.top_k, role=req.role)

    answer_parts = []
    try:
        for tok in call_llm_groq(req.question, hits, role=req.role):
            answer_parts.append(tok)
    except Exception as e:
        return {
            "question": req.question,
            "top_k": req.top_k,
            "role": req.role,
            "answer": "",
            "sources": [{"score": h.get("score"), "meta": h.get("meta")} for h in hits],
            "error": str(e)
        }

    return {
        "question": req.question,
        "top_k": req.top_k,
        "role": req.role,
        "answer": "".join(answer_parts),
        "sources": [{"score": h.get("score"), "meta": h.get("meta")} for h in hits]
    }

@app.post("/ask_stream")  # stream tokens
def ask_stream(req: AskReq):
    hits = retrieve_with_fallback(req.question, req.top_k, role=req.role)

    def gen():
        try:
            for tok in call_llm_groq(req.question, hits, role=req.role):
                yield tok
        except Exception as e:
            yield f"\n[LLM error] {str(e)}"

    return StreamingResponse(gen(), media_type="text/plain")

# --- run uvicorn in notebook safely ---
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_server, daemon=True).start()
print("✅ API running at http://127.0.0.1:8000  (docs: /docs)")


✅ API running at http://127.0.0.1:8000  (docs: /docs)


In [49]:
import os
print("GROQ_API_KEY exists:", bool(os.getenv("GROQ_API_KEY")))


GROQ_API_KEY exists: True


In [50]:
#demo

In [51]:
import requests
with requests.post("http://127.0.0.1:8000/ask_stream", json={"question":"có những lỗi gì trong hệ thống", "top_k":3, "role":"admin"}, stream=True) as r:
    for chunk in r.iter_content(chunk_size=None):
        if chunk:
            print(chunk.decode("utf-8"), end="")


INFO:     Started server process [30824]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51865 - "POST /ask_stream HTTP/1.1" 200 OK
Có những lỗi sau trong hệ thống:

- Lỗi xác thực tài khoản (Email hoặc mật khẩu không chính xác hoặc tài khoản bị khóa) - [1]
- Lỗi hệ thống khi test hê thống Log lỗi AEMS - [2]
- Lỗi xác thực Google (Access was denied by the resource owner or by the remote server) - [3]

In [52]:
# Demo: show a few KB chunks and LOG summary chunks
print("=== KB SAMPLE ===")
for i, item in enumerate(kb_chunks[:3]):
    m = item["meta"]
    print("="*80)
    print("CHUNK", i, "| type:", m.get("type"), "| access:", m.get("access"))
    print("event_id:", m.get("event_id"))
    print(item["text"][:500])

print("\n=== LOG SUMMARY SAMPLE ===")
for i, item in enumerate(log_chunks[:3]):
    m = item["meta"]
    print("="*80)
    print("CHUNK", i, "| type:", m.get("type"), "| access:", m.get("access"))
    print("exception_type:", m.get("exception_type"), "| status_code:", m.get("status_code"), "| count:", m.get("count"))
    print(item["text"][:500])


=== KB SAMPLE ===
CHUNK 0 | type: event | access: user
event_id: 0D585A2C-40AA-4C0C-B4E9-B097AF559C4F
EVENT OVERVIEW
Title: AI Workshop 2026
Description: Workshop giới thiệu về AI và Machine Learning.
Time: 2026-03-10 08:00:00 - 2026-03-10 11:00:00
Status: Published

LOCATION DETAILS
- Room A101
  Address: Building A - Floor 1
  Capacity: 150
  Type: 1
- Room B202
  Address: Building B - Floor 2
  Capacity: 100
  Type: 1

=== LOG SUMMARY SAMPLE ===
CHUNK 0 | type: system_log_summary | access: admin
exception_type: ArgumentException | status_code: 500.0 | count: 346
SYSTEM ERROR SUMMARY
StatusCode: 500.0
ExceptionType: ArgumentException
Source: /Home/Error
Message: The 'ClientId' option must be provided. (Parameter 'ClientId') | Inner Cause: The 'ClientId' option must be provided. (Parameter 'ClientId')
Count: 346
FirstAt: 2026-02-02 15:29:56.349393
LastAt: 2026-02-03 09:18:51.044523
CHUNK 1 | type: system_log_summary | access: admin
exception_type: ArgumentException | status_code: nan 

In [53]:
print("kb_index.ntotal  =", kb_index.ntotal)
print("log_index.ntotal =", log_index.ntotal)
print("len(kb_chunks)   =", len(kb_chunks))
print("len(log_chunks)  =", len(log_chunks))


kb_index.ntotal  = 1
log_index.ntotal = 21
len(kb_chunks)   = 1
len(log_chunks)  = 21


In [54]:
# Quick test with LLM streaming - Now with valid API key
import logging
logging.basicConfig(level=logging.INFO)

q = "có sự kiện nào không?"
print("=" * 80)
print("Retrieving documents...")
print("=" * 80)

hits_user  = retrieve_with_fallback(q, top_k=3, role="user")
hits_admin = retrieve_with_fallback("trong hệ thống có lỗi gì không?", top_k=3, role="admin")

print(f"User hits: {len(hits_user)} | Admin hits: {len(hits_admin)}")

print("\n" + "=" * 80)
print("USER QUESTION:", q)
print("=" * 80)

try:
    print("USER ANSWER:")
    output = ""
    for tok in call_llm_groq(q, hits_user, role="user"):
        print(tok, end="", flush=True)
        output += tok
    print("\n")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")

print("=" * 80)
print("ADMIN QUESTION: Có lỗi gì trong hệ thống?")
print("=" * 80)

try:
    print("ADMIN ANSWER:")
    output = ""
    for tok in call_llm_groq("trong hệ thống có lỗi gì không?", hits_admin, role="admin"):
        print(tok, end="", flush=True)
        output += tok
    print("\n")
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")


Retrieving documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.92it/s]

User hits: 1 | Admin hits: 3

USER QUESTION: có sự kiện nào không?
USER ANSWER:


Có sự kiện "AI Workshop 2026" [1].

ADMIN QUESTION: Có lỗi gì trong hệ thống?
ADMIN ANSWER:
Có lỗi trong hệ thống, cụ thể là:

- Lỗi Email hoặc mật khẩu không chính xác hoặc tài khoản bị khóa tại AuthController.Login (mục [1], [3]).
- Lỗi hệ thống không xác định tại nguồn "/" (mục [2]).



In [55]:
# ===== DEMO: Test API without GROQ API Key =====
print("=" * 80)
print("DEMO: Testing RAG System (Mock Response - no API key needed)")
print("=" * 80)

def call_llm_mock(question: str, hits: list, role: str = "user") -> str:
    """Mock LLM response for demo/testing (không cần API key)."""
    role = (role or "user").lower().strip()
    
    # Filter by access
    filtered = [h for h in (hits or []) if can_access(h.get("meta", {}), role)]
    
    if not filtered:
        return "Không tìm thấy thông tin trong dữ liệu."
    
    # Build context
    context = _format_context(filtered, max_chars_each=500)
    
    # Mock response based on question
    if role == "admin":
        return f"""[Admin Mode] Dựa trên dữ liệu hệ thống:
        
{context[:300]}...

Tìm thấy {len(filtered)} mục liên quan. Hãy xem chi tiết trên dashboard."""
    
    else:  # user
        return f"""[User Mode] Thông tin về sự kiện:

{context[:300]}...

Tìm thấy {len(filtered)} sự kiện phù hợp với yêu cầu của bạn."""

# Test
q = "có sự kiện nào không?"
hits = retrieve_with_fallback(q, top_k=3, role="user")

print("\n📝 Question:", q)
print("\n✅ Mock Response:")
print(call_llm_mock(q, hits, role="user"))




DEMO: Testing RAG System (Mock Response - no API key needed)


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.67it/s]


📝 Question: có sự kiện nào không?

✅ Mock Response:
[User Mode] Thông tin về sự kiện:

[1] type=event access=user id=0D585A2C-40AA-4C0C-B4E9-B097AF559C4F time=2026-02-13 16:59:10.756666
EVENT OVERVIEW
Title: AI Workshop 2026
Description: Workshop giới thiệu về AI và Machine Learning.
Time: 2026-03-10 08:00:00 - 2026-03-10 11:00:00
Status: Published

LOCATION DETAILS
- Room A101
  Addr...

Tìm thấy 1 sự kiện phù hợp với yêu cầu của bạn.
